扑克牌游戏
首先游戏规则就是存在52张牌（没有大小王），需要将52张牌发到4个玩家手中，每个玩家手上存在13张牌，按照4中花色的黑桃、红心、草花、方块的顺序以及点数从小到大排列

首先面对问题，我们抽象出了一个是玩家类、扑克类以及牌类。
从最基础的牌类进行创建，首先是牌有两种特性，一个是花色一个是点数。花色使用枚举，但是Python本身没有枚举这种类型，如果使用0~3代码可读性较差，所以采用enum模块的Enum类创建枚举类型。
创建Card类时牌需要输入两种属性，

In [31]:
from enum import Enum
import random

class Suite(Enum):
    """花色(枚举)"""
    SPADE, HEART, CLUB, DIAMOND = range(4)

class Card:
    """牌"""

    def __init__(self, suite, face):
        self.suite = suite
        self.face = face
    
    def __repr__(self):
        suites = '♠♥♣♦'
        faces = ['', 'A', '2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K']
        return f'{suites[self.suite.value]}{faces[self.face]}'  # 返回牌的花色和点数

    def __lt__(self, other):
        if self.suite == other.suite:
            return self.face < other.face   # 花色相同比较点数的大小
        return self.suite.value < other.suite.value   # 花色不同比较花色对应的值

class Poker:
    """扑克"""
    def __init__(self):
        self.cards = [Card(suite, face) 
                      for suite in Suite
                      for face in range(1, 14)]  # 52张牌构成的列表
        self.current = 0  # 记录发牌位置的属性

    def shuffle(self):
        """洗牌"""
        self.current = 0
        random.shuffle(self.cards)  # 通过random模块的shuffle函数实现随机乱序

    def deal(self):
        """发牌"""
        card = self.cards[self.current]
        self.current += 1
        return card
    
    @property
    def has_next(self):
        """还有没有牌可以发"""
        return self.current < len(self.cards)

class Player:
    """玩家"""

    def __init__(self, name):
        self.name = name
        self.cards = []  # 玩家手上的牌

    def get_one(self, card):
        """摸牌"""
        self.cards.append(card)

    def arrange(self):
        """整理手上的牌"""
        self.cards.sort()

poker = Poker()
poker.shuffle()
players = [Player('东邪'), Player('西毒'), Player('南帝'), Player('北丐')]
# 将牌轮流发到每个玩家手上每人13张牌
for _ in range(13):
    for player in players:
        player.get_one(poker.deal())
# 玩家整理手上的牌输出名字和手牌
for player in players:
    player.arrange()
    print(f'{player.name}: ', end='')
    print(player.cards)

东邪: [♠10, ♥4, ♥6, ♥7, ♥10, ♣A, ♣4, ♣5, ♣6, ♣7, ♣9, ♣K, ♦K]
西毒: [♠A, ♠5, ♠9, ♥2, ♥3, ♥9, ♣3, ♣10, ♣J, ♦5, ♦7, ♦9, ♦J]
南帝: [♠4, ♠6, ♠Q, ♠K, ♥8, ♥Q, ♣2, ♣8, ♣Q, ♦3, ♦4, ♦6, ♦Q]
北丐: [♠2, ♠3, ♠7, ♠8, ♠J, ♥A, ♥5, ♥J, ♥K, ♦A, ♦2, ♦8, ♦10]


在这个程序中之前存在一个问题就是比较Card的大小时，其中存在两个类型一个是花色一个是数字大小，调用sort进行排序时不知道如何比较，就是Card1 < Card2 这种情况。所以需要对这个 < 进行运算符重载。Python 中要实现对<运算符的重载，需要在类中添加一个名为__lt__的魔术方法。很显然，魔术方法__lt__中的lt是英文单词“less than”的缩写，以此类推，魔术方法__gt__对应>运算符，魔术方法__le__对应<=运算符，__ge__对应>=运算符，__eq__对应==运算符，__ne__对应!=运算符。

接着我们对这个游戏进行升级，改变游戏规则，改为斗地主规则，首先是添加牌小王以及大王，
然后玩家数改为三人，每人17张牌，然后给地主多发三张牌。

In [44]:
from enum import Enum
import random

class Suite(Enum):
    """花色(枚举)"""
    SPADE, HEART, CLUB, DIAMOND, SMALL_JOKER, BIG_JOKER = range(6)

class Card:
    """牌"""

    def __init__(self, suite, face):
        self.suite = suite
        self.face = face
    
    def __repr__(self):
        suites = '♠♥♣♦🃏🂿'
        faces = ['', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'A', '2']
        return f'{suites[self.suite.value]}{faces[self.face]}'  # 返回牌的花色和点数

    def __lt__(self, other):
        if self.suite == Suite.SMALL_JOKER or self.suite == Suite.BIG_JOKER or other.suite == Suite.SMALL_JOKER or other.suite == Suite.BIG_JOKER:
            return self.suite.value < other.suite.value  # 小王和大王比较花色对应的值
        elif self.face == other.face:
            return self.suite.value < other.suite.value   # 点数相同对比花色的大小
        return self.face < other.face   # 花色不同比较点数的大小

class Poker:
    """扑克"""
    def __init__(self):
        self.cards = [Card(suite, face) 
                      for suite in Suite if suite != Suite.SMALL_JOKER and suite != Suite.BIG_JOKER 
                      for face in range(1, 14)]  # 52张牌构成的列表
        self.cards.append(Card(Suite.SMALL_JOKER, 0))  # 添加一张小王
        self.cards.append(Card(Suite.BIG_JOKER, 0))  # 添加一张大王
        self.current = 0  # 记录发牌位置的属性

    def shuffle(self):
        """洗牌"""
        self.current = 0
        random.shuffle(self.cards)  # 通过random模块的shuffle函数实现随机乱序

    def deal(self):
        """发牌"""
        card = self.cards[self.current]
        self.current += 1
        return card
    
    @property
    def has_next(self):
        """还有没有牌可以发"""
        return self.current < len(self.cards)

class Player:
    """玩家"""

    def __init__(self, name):
        self.name = name
        self.cards = []  # 玩家手上的牌

    def get_one(self, card):
        """摸牌"""
        self.cards.append(card)

    def arrange(self):
        """整理手上的牌"""
        self.cards.sort()

poker = Poker()
poker.shuffle()
players = [Player('地主'), Player('农民2'), Player('农民1')]
# 将牌轮流发到每个玩家手上每人13张牌
for _ in range(17):
    for player in players:
        player.get_one(poker.deal())
for  _ in range(3):
    players[0].get_one(poker.deal())
# 玩家整理手上的牌输出名字和手牌
for player in players:
    player.arrange()
    print(f'{player.name}: ', end='')
    print(player.cards)

地主: [♠3, ♥3, ♣3, ♦3, ♦4, ♦5, ♥6, ♣6, ♦6, ♠7, ♠8, ♥8, ♦8, ♠9, ♣9, ♠10, ♣10, ♠K, ♦K, ♠2]
农民2: [♠4, ♥4, ♠5, ♥5, ♣7, ♦7, ♣8, ♥9, ♥J, ♥Q, ♣Q, ♥K, ♣K, ♥A, ♦A, ♥2, 🃏]
农民1: [♣4, ♣5, ♠6, ♥7, ♦9, ♥10, ♦10, ♠J, ♣J, ♦J, ♠Q, ♦Q, ♠A, ♣A, ♣2, ♦2, 🂿]
